# Advanced Problems with Solutions: Decorating Classes in Python

This notebook contains advanced practice problems on class decorators, monkey patching, runtime class modification, comparison methods, `NotImplemented`, and `functools.total_ordering`.

Each problem includes a full solution and test cases.

## Problem 1 — Safe Debug Class Decorator

Write a class decorator called `debug_info` that adds a `.debug()` method to any decorated class.

The `.debug()` method should return a dictionary containing:

- the class name
- the memory address of the instance, using `hex(id(self))`
- all instance attributes from `vars(self)`

Best-practice requirements:

1. The decorator must not overwrite an existing `.debug()` method.
2. If the class already defines `.debug()`, raise a `TypeError`.
3. The decorator must return the same class object it receives.

In [1]:
def debug_info(cls):
    if hasattr(cls, 'debug'):
        raise TypeError(f'{cls.__name__} already defines debug()')

    def debug(self):
        return {
            'class': self.__class__.__name__,
            'id': hex(id(self)),
            'attributes': dict(vars(self))
        }

    cls.debug = debug
    return cls


@debug_info
class User:
    def __init__(self, username, active=True):
        self.username = username
        self.active = active


u = User('simeon')
u.debug()

{'class': 'User',
 'id': '0x23246c40d70',
 'attributes': {'username': 'simeon', 'active': True}}

### Solution Explanation

A class decorator receives the class object after the class body has been executed. We attach a new method directly to the class, which makes it available to all instances.

The important defensive step is checking whether the class already has a `debug` attribute. Silently overwriting existing methods is dangerous because it can break the original class behavior.

In [2]:
# Test: the decorator should reject a class that already defines debug()
try:
    @debug_info
    class BadUser:
        def debug(self):
            return 'already exists'
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError BadUser already defines debug()


## Problem 2 — Class Decorator Factory for Validation

Create a decorator factory called `require_attrs(*names)`.

It should return a class decorator that wraps the class's `__init__` method. After the original `__init__` runs, the wrapper should verify that the instance has all required attributes.

If any required attribute is missing, raise an `AttributeError` listing the missing attributes.

Best-practice requirements:

1. Preserve the original `__init__` metadata with `functools.wraps`.
2. Do not skip the original `__init__`.
3. Return `NotImplemented` nowhere; this problem is about validation, not comparison.

In [3]:
from functools import wraps


def require_attrs(*names):
    def decorator(cls):
        original_init = cls.__init__

        @wraps(original_init)
        def new_init(self, *args, **kwargs):
            original_init(self, *args, **kwargs)
            missing = [name for name in names if not hasattr(self, name)]
            if missing:
                raise AttributeError(
                    f'{cls.__name__} missing required attributes: {missing}'
                )

        cls.__init__ = new_init
        return cls

    return decorator


@require_attrs('name', 'email')
class Customer:
    def __init__(self, name, email):
        self.name = name
        self.email = email


c = Customer('Ada', 'ada@example.com')
vars(c)

{'name': 'Ada', 'email': 'ada@example.com'}

In [4]:
# Test: missing required attribute
try:
    @require_attrs('name', 'email')
    class BrokenCustomer:
        def __init__(self, name):
            self.name = name

    BrokenCustomer('Ada')
except AttributeError as ex:
    print(type(ex).__name__, ex)

AttributeError BrokenCustomer missing required attributes: ['email']


### Solution Explanation

`require_attrs` is a decorator factory because it receives configuration first and returns the actual class decorator.

The actual decorator receives the class, stores its original `__init__`, replaces it with a wrapper, and returns the same class.

`functools.wraps` is useful here because the replacement initializer still represents the original initializer logically.

## Problem 3 — Implement a Robust `complete_ordering` Decorator

Write a class decorator called `complete_ordering`.

The decorated class must define `__eq__` and `__lt__`. Your decorator should add:

- `__le__`
- `__gt__`
- `__ge__`

Best-practice requirements:

1. Raise `TypeError` if the class does not define both `__eq__` and `__lt__` directly.
2. Use `type(self).__lt__(self, other)` and `type(self).__eq__(self, other)` instead of inline operators.
3. Correctly propagate `NotImplemented`.
4. Avoid infinite reflection loops.

In [5]:
def complete_ordering(cls):
    if '__eq__' not in cls.__dict__ or '__lt__' not in cls.__dict__:
        raise TypeError(
            f'{cls.__name__} must define __eq__ and __lt__ directly'
        )

    def __le__(self, other):
        lt = type(self).__lt__(self, other)
        if lt is NotImplemented:
            return NotImplemented
        if lt:
            return True

        eq = type(self).__eq__(self, other)
        if eq is NotImplemented:
            return NotImplemented
        return eq

    def __gt__(self, other):
        le = __le__(self, other)
        if le is NotImplemented:
            return NotImplemented
        return not le

    def __ge__(self, other):
        lt = type(self).__lt__(self, other)
        if lt is NotImplemented:
            return NotImplemented
        return not lt

    cls.__le__ = __le__
    cls.__gt__ = __gt__
    cls.__ge__ = __ge__
    return cls

In [6]:
from math import sqrt


@complete_ordering
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __abs__(self):
        return sqrt(self.x ** 2 + self.y ** 2)

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __lt__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return abs(self) < abs(other)

    def __repr__(self):
        return f'Point({self.x}, {self.y})'


p1 = Point(1, 1)
p2 = Point(3, 4)
p3 = Point(3, 4)

p1 < p2, p1 <= p2, p2 >= p3, p2 > p1, p1 > p2

(True, True, True, True, False)

### Solution Explanation

The key issue is avoiding expressions like `self < other` inside the generated comparison methods. Those inline operators may trigger Python's reflected comparison machinery, which can create confusing behavior or recursion.

Calling `type(self).__lt__(self, other)` directly lets us inspect whether the method returns `NotImplemented` and propagate that result correctly.

In [7]:
# Test: incompatible comparison should fail cleanly
try:
    print(Point(1, 2) <= 10)
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError '<=' not supported between instances of 'Point' and 'int'


## Problem 4 — Add a Cached Computed Property with a Class Decorator

Write a decorator factory called `cached_method_as_property(method_name, property_name=None)`.

It should convert a zero-argument instance method into a cached property.

Example:

```python
@cached_method_as_property('area')
class Circle:
    def area(self):
        return 3.14 * self.radius ** 2
```

After decoration, `circle.area` should compute once and then return the cached value.

Best-practice requirements:

1. The original method must exist.
2. The original method must be callable.
3. Store the cached value on the instance.
4. Support a custom property name.

In [8]:
def cached_method_as_property(method_name, property_name=None):
    def decorator(cls):
        if not hasattr(cls, method_name):
            raise AttributeError(f'{cls.__name__} has no attribute {method_name!r}')

        method = getattr(cls, method_name)
        if not callable(method):
            raise TypeError(f'{method_name!r} must be callable')

        prop_name = property_name or method_name
        cache_name = f'_cached_{prop_name}'

        def getter(self):
            if cache_name not in vars(self):
                setattr(self, cache_name, method(self))
            return getattr(self, cache_name)

        setattr(cls, prop_name, property(getter))
        return cls

    return decorator


@cached_method_as_property('compute_area', 'area')
class Circle:
    def __init__(self, radius):
        self.radius = radius
        self.calls = 0

    def compute_area(self):
        self.calls += 1
        return 3.14159 * self.radius ** 2


circle = Circle(10)
circle.area, circle.area, circle.calls

(314.159, 314.159, 1)

### Solution Explanation

The decorator replaces or creates a class-level property. The property getter stores the computed value in the instance dictionary the first time it is accessed.

This is useful when the computed value is expensive and the object's relevant state does not change after initialization.

## Problem 5 — Decorator for Immutable Instances After Initialization

Create a class decorator called `freeze_after_init`.

After an object is initialized, assigning new or existing attributes should raise an `AttributeError`.

Best-practice requirements:

1. The original `__init__` must still run normally.
2. Attribute assignment should be allowed during initialization.
3. Attribute assignment should be blocked after initialization.
4. Preserve the original `__init__` metadata with `functools.wraps`.

In [9]:
from functools import wraps


def freeze_after_init(cls):
    original_init = cls.__init__
    original_setattr = cls.__setattr__

    @wraps(original_init)
    def __init__(self, *args, **kwargs):
        object.__setattr__(self, '_is_frozen', False)
        original_init(self, *args, **kwargs)
        object.__setattr__(self, '_is_frozen', True)

    def __setattr__(self, name, value):
        if getattr(self, '_is_frozen', False):
            raise AttributeError(
                f'{type(self).__name__} instances are frozen; cannot set {name!r}'
            )
        original_setattr(self, name, value)

    cls.__init__ = __init__
    cls.__setattr__ = __setattr__
    return cls


@freeze_after_init
class Config:
    def __init__(self, host, port):
        self.host = host
        self.port = port


cfg = Config('localhost', 8000)
vars(cfg)

{'_is_frozen': True, 'host': 'localhost', 'port': 8000}

In [10]:
try:
    cfg.port = 9000
except AttributeError as ex:
    print(type(ex).__name__, ex)

AttributeError Config instances are frozen; cannot set 'port'


### Solution Explanation

This decorator works by temporarily allowing mutation while `__init__` runs. After initialization, `_is_frozen` is set to `True`.

The custom `__setattr__` checks that flag before permitting assignment.

Using `object.__setattr__` for the internal flag avoids accidentally calling the decorated class's own `__setattr__` too early.

## Problem 6 — Registration Decorator for Plugin Classes

Create a decorator factory called `register_plugin(registry, name=None)`.

It should register decorated classes in a dictionary.

If `name` is not provided, use the class name.

Best-practice requirements:

1. Do not instantiate the class during registration.
2. Reject duplicate plugin names.
3. Return the original class unchanged.
4. Show that the registered class can be instantiated later.

In [11]:
def register_plugin(registry, name=None):
    def decorator(cls):
        plugin_name = name or cls.__name__
        if plugin_name in registry:
            raise KeyError(f'Plugin {plugin_name!r} is already registered')
        registry[plugin_name] = cls
        return cls
    return decorator


plugins = {}


@register_plugin(plugins)
class CsvExporter:
    def export(self, rows):
        return '\n'.join(','.join(map(str, row)) for row in rows)


@register_plugin(plugins, name='json')
class JsonExporter:
    def export(self, rows):
        import json
        return json.dumps(rows)


plugins

{'CsvExporter': __main__.CsvExporter, 'json': __main__.JsonExporter}

In [12]:
exporter_cls = plugins['json']
exporter = exporter_cls()
exporter.export([{'name': 'Ada'}, {'name': 'Linus'}])

'[{"name": "Ada"}, {"name": "Linus"}]'

### Solution Explanation

A registration decorator is a common real-world use of class decorators. It records class objects in a registry without changing their behavior.

The class is not instantiated during decoration. This keeps registration cheap and avoids side effects.

## Problem 7 — Add JSON Serialization to Classes

Create a class decorator called `json_serializable`.

It should add a `.to_json()` method to the decorated class.

The method should serialize the instance dictionary to a JSON string.

Best-practice requirements:

1. Do not overwrite an existing `.to_json()` method.
2. Use `json.dumps`.
3. Support non-JSON-native values by using `default=str`.
4. Sort keys for deterministic output.

In [13]:
import json
from datetime import datetime, timezone


def json_serializable(cls):
    if hasattr(cls, 'to_json'):
        raise TypeError(f'{cls.__name__} already defines to_json()')

    def to_json(self):
        return json.dumps(vars(self), default=str, sort_keys=True)

    cls.to_json = to_json
    return cls


@json_serializable
class Event:
    def __init__(self, name, created_at):
        self.name = name
        self.created_at = created_at


event = Event('deploy', datetime.now(timezone.utc))
event.to_json()

'{"created_at": "2026-05-22 15:24:32.920839+00:00", "name": "deploy"}'

### Solution Explanation

This decorator adds a reusable behavior to any class whose instances store state in `__dict__`.

`default=str` is a practical safeguard for objects such as `datetime`, which are not JSON serializable by default.

## Problem 8 — Compare Custom `complete_ordering` with `functools.total_ordering`

Use Python's built-in `functools.total_ordering` to implement a sortable `Version` class.

A version has:

- `major`
- `minor`
- `patch`

Versions should compare lexicographically by `(major, minor, patch)`.

Best-practice requirements:

1. Implement `__eq__`.
2. Implement only one ordering method, such as `__lt__`.
3. Return `NotImplemented` for incompatible types.
4. Use `@total_ordering`.

In [14]:
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, major, minor, patch):
        self.major = major
        self.minor = minor
        self.patch = patch

    def _key(self):
        return self.major, self.minor, self.patch

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._key() == other._key()

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._key() < other._key()

    def __repr__(self):
        return f'Version({self.major}, {self.minor}, {self.patch})'


v1 = Version(1, 2, 0)
v2 = Version(1, 2, 3)
v3 = Version(2, 0, 0)

v1 < v2, v2 <= v3, v3 > v1, sorted([v3, v1, v2])

(True, True, True, [Version(1, 2, 0), Version(1, 2, 3), Version(2, 0, 0)])

### Solution Explanation

`total_ordering` is the production-ready version of the custom ordering decorator idea.

When a class defines `__eq__` and one ordering method, `total_ordering` fills in the rest.

Returning `NotImplemented` is important because it gives Python a chance to handle reflected operations or raise a proper `TypeError`.

## Problem 9 — A Decorator That Adds Repr from Constructor Attributes

Create a class decorator factory called `auto_repr(*attrs)`.

It should add a `__repr__` method that formats selected instance attributes.

Example output:

```python
Book(title='Python', pages=400)
```

Best-practice requirements:

1. Do not overwrite an existing `__repr__` unless `overwrite=True` is explicitly passed.
2. Use `!r` formatting for attribute values.
3. Raise an informative error if an attribute is missing.
4. Support keyword-only `overwrite=False`.

In [15]:
def auto_repr(*attrs, overwrite=False):
    def decorator(cls):
        if '__repr__' in cls.__dict__ and not overwrite:
            raise TypeError(
                f'{cls.__name__} already defines __repr__; use overwrite=True'
            )

        def __repr__(self):
            parts = []
            for attr in attrs:
                if not hasattr(self, attr):
                    raise AttributeError(
                        f'{type(self).__name__} instance has no attribute {attr!r}'
                    )
                parts.append(f'{attr}={getattr(self, attr)!r}')
            return f'{type(self).__name__}(' + ', '.join(parts) + ')'

        cls.__repr__ = __repr__
        return cls

    return decorator


@auto_repr('title', 'pages')
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages


Book('Python Internals', 500)

Book(title='Python Internals', pages=500)

### Solution Explanation

This decorator factory is reusable because the caller controls which attributes appear in the generated representation.

The decorator checks for an existing `__repr__` directly in `cls.__dict__`, which avoids rejecting inherited methods unnecessarily.

## Problem 10 — Compose Multiple Class Decorators Safely

Create a class that uses several decorators together:

- `@debug_info`
- `@json_serializable`
- `@auto_repr(...)`

Then explain the order in which the decorators are applied.

Best-practice requirements:

1. Demonstrate that all added methods work.
2. Explain why decorator order matters.
3. Avoid method-name collisions.

In [16]:
@debug_info
@json_serializable
@auto_repr('sku', 'price')
class Product:
    def __init__(self, sku, price):
        self.sku = sku
        self.price = price


product = Product('ABC-123', 19.99)

repr(product), product.to_json(), product.debug()

("Product(sku='ABC-123', price=19.99)",
 '{"price": 19.99, "sku": "ABC-123"}',
 {'class': 'Product',
  'id': '0x23246c430e0',
  'attributes': {'sku': 'ABC-123', 'price': 19.99}})

### Solution Explanation

Decorator application happens from bottom to top.

This:

```python
@debug_info
@json_serializable
@auto_repr('sku', 'price')
class Product:
    pass
```

is equivalent to:

```python
Product = debug_info(json_serializable(auto_repr('sku', 'price')(Product)))
```

Order matters when decorators depend on methods added by earlier decorators or when two decorators try to add the same method name.

# Summary

Class decorators are useful when you want to modify, validate, register, or enrich classes at definition time.

Best practices:

- Return the class object from the decorator.
- Avoid silently overwriting existing methods.
- Preserve metadata when wrapping methods.
- Use `NotImplemented` correctly in comparison methods.
- Prefer standard tools such as `functools.total_ordering` when available.
- Be careful when monkey patching special methods.
- Keep decorators reusable and focused.